<a href="https://colab.research.google.com/github/SaiSaraswathi06/AIBootCamp-Task2B/blob/main/Day2_ResumeExtractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai pydantic

import os
import getpass

if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [2]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [3]:
from google import genai
from pydantic import ValidationError

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

def extract_resume(raw_text: str, max_retries: int = 1):

    for attempt in range(max_retries + 1):

        try:

            response = client.models.generate_content(
                model="gemini-2.5-flash",

                contents=f"""
Extract a Resume JSON from this text.

Return ONLY JSON.

{raw_text}
""",

                config={
                    "response_mime_type":"application/json",
                    "response_schema": Resume.model_json_schema()
                }
            )

            return Resume.model_validate_json(
                response.text
            )

        except ValidationError as e:

            if attempt == max_retries:
                raise

            fix_prompt = f"""
Fix this JSON.

Errors:
{e}

Original JSON:
{response.text}
"""

            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=fix_prompt,

                config={
                    "response_mime_type":"application/json",
                    "response_schema": Resume.model_json_schema()
                }
            )

            return Resume.model_validate_json(
                response.text
            )

In [4]:
with open("sample_resumes.txt","r") as f:

    resumes = [
        r.strip()
        for r in f.read().split("---")
        if r.strip()
    ]

print("Loaded", len(resumes), "resumes")

results = []

for i,resume in enumerate(resumes[:3]):

    try:

        parsed = extract_resume(resume)

        results.append(parsed)

        print(
            f"\nResume {i+1}: "
            f"{parsed.name} | "
            f"{len(parsed.skills)} skills | "
            f"{parsed.experience_years} years"
        )

    except Exception as e:

        print(
            f"\nResume {i+1} FAILED:",
            str(e)
        )

if results:

    print("\nFULL FIRST RESULT\n")

    print(
        results[0].model_dump_json(
            indent=2
        )
    )

Loaded 3 resumes

Resume 1: Ravi Kumar | 6 skills | 1.0 years

Resume 2: Sneha Reddy | 6 skills | 0.5 years

Resume 3: Arun Pillai | 9 skills | 1.0 years

FULL FIRST RESULT

{
  "name": "Ravi Kumar",
  "email": "ravi@gmail.com",
  "phone": "9876543210",
  "education": [
    {
      "degree": "B.Tech CSE",
      "institution": "Aditya Engineering College",
      "year": 2024
    }
  ],
  "skills": [
    "Python",
    "Java",
    "SQL",
    "HTML",
    "CSS",
    "Git"
  ],
  "projects": [
    "Student Management System"
  ],
  "experience_years": 1.0
}


In [5]:
try:

    bad = extract_resume("")

    print(
        "Unexpected success",
        bad.model_dump_json()
    )

except Exception as e:

    print(
        "Caught gracefully:",
        type(e).__name__
    )

    print(
        "Message:",
        str(e)[:200]
    )

Unexpected success {"name":"","email":"","phone":null,"education":[],"skills":[],"projects":[],"experience_years":0.0}
